In [2]:
import torch
from torch import nn 
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor 

In [3]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print(device)

cpu


In [4]:
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

batch_size = 64
train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

100%|██████████| 26.4M/26.4M [00:24<00:00, 1.07MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 211kB/s]
100%|██████████| 4.42M/4.42M [00:04<00:00, 1.08MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 34.3MB/s]


In [5]:
class NeuralNetwork(nn.Module):
    
    def __init__(self):
        
        super().__init__()
        
        # Flatten: reshapes input (batch_size, 1, 28, 28) -> (batch_size, 784)
        self.flatten = nn.Flatten()
        
        # Sequential container chaining linear transforms and activations
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),  # Z = XW + b (W: 784x512, b: 512)
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)         # Logits output (raw unnormalized scores)
        )

    def forward(self, x):
        x = self.flatten(x)  
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)

In [6]:
# CrossEntropyLoss combines nn.LogSoftmax() and nn.NLLLoss()
loss_fn = nn.CrossEntropyLoss()

# Stochastic Gradient Descent optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [7]:
def train(dataloader, model, loss_fn, optimizer):
    model.train()  # Sets model mode to training (activates dropout/batchnorm behavior)
    size = len(dataloader.dataset)
    
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # 1. Forward Pass (Compute predictions)
        pred = model(X)
        loss = loss_fn(pred, y)

        # 2. Backward Pass (Compute gradients)
        loss.backward()        # Autograd computes dL/dW and dL/db for every parameter

        # 3. Parameter Update
        optimizer.step()       # W = W - lr * grad

        optimizer.zero_grad()  # Clear gradients from previous iteration
        
        if batch % 100 == 0:
            loss_val, current = loss.item(), batch * len(X)
            print(f"loss: {loss_val:>7f}  [{current:>5d}/{size:>5d}]")

def test(dataloader, model, loss_fn):
    model.eval()  # Sets model mode to evaluation
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Disable autograd tracking to save memory and compute during inference
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f}\n")

In [8]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Quickstart pipeline complete.")

Epoch 1
-------------------------------
loss: 2.310275  [    0/60000]
loss: 2.289360  [ 6400/60000]
loss: 2.280813  [12800/60000]
loss: 2.266774  [19200/60000]
loss: 2.252095  [25600/60000]
loss: 2.250620  [32000/60000]
loss: 2.229234  [38400/60000]
loss: 2.183840  [44800/60000]
loss: 2.194006  [51200/60000]
loss: 2.173536  [57600/60000]
Test Accuracy: 39.6%, Avg loss: 2.174885

Epoch 2
-------------------------------
loss: 2.174019  [    0/60000]
loss: 2.170591  [ 6400/60000]
loss: 2.150707  [12800/60000]
loss: 2.113997  [19200/60000]
loss: 2.070041  [25600/60000]
loss: 2.046949  [32000/60000]
loss: 2.029832  [38400/60000]
loss: 1.984847  [44800/60000]
loss: 1.969758  [51200/60000]
loss: 1.949003  [57600/60000]
Test Accuracy: 57.1%, Avg loss: 1.925433

Epoch 3
-------------------------------
loss: 1.917132  [    0/60000]
loss: 1.843183  [ 6400/60000]
loss: 1.875955  [12800/60000]
loss: 1.794369  [19200/60000]
loss: 1.765896  [25600/60000]
loss: 1.779503  [32000/60000]
loss: 1.733187  

In [ ]:
for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Requires Grad: {param.requires_grad}")


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Requires Grad: True
Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Requires Grad: True
Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Requires Grad: True
Layer: linear_relu_stack.2.bias | Size: torch.Size([512]) | Requires Grad: True
Layer: linear_relu_stack.4.weight | Size: torch.Size([10, 512]) | Requires Grad: True
Layer: linear_relu_stack.4.bias | Size: torch.Size([10]) | Requires Grad: True


In [11]:
print(model.linear_relu_stack[0].weight.grad)

None
